In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from pydantic import BaseModel
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from zoomcamp.evaluation_utils import llm_structured

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

print(len(documents))

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
class Questions(BaseModel):
    questions: list[str]

In [ ]:
load_dotenv()
openai_client = OpenAI()

## Q1. Generating questions

In [ ]:
first_3_docs = documents[:3]
input_tokens = []

for doc in first_3_docs:

    user_prompt = json.dumps(doc)
    messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
    ]

    result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
    )

    input_tokens.append(usage.input_tokens)

print(f'Average input token usage: {sum(input_tokens) / len(input_tokens)}')

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))
print(chunks[0])

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
df_ground_truth = pd.read_csv('ground-truth.csv')
ground_truth = df_ground_truth.to_dict(orient='records')
q = ground_truth[0]['question']

print(len(ground_truth))
print(q)

## Q2. First result with text search

In [ ]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

text_results = text_search(q)
text_results[0]['filename']

## Q3. First result with vector search

In [ ]:
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [chunk['content'] for chunk in chunks]

batch_size = 50
vectors = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [ ]:
def vector_search(query, num_results=5):
    query_vector = model.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [ ]:
vector_results = vector_search(q)
vector_results[0]['filename']

## Q4. Evaluating text search

In [ ]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(ground_truth, text_search)

## Q5. Evaluating vector search

In [ ]:
evaluate(ground_truth, vector_search)

## Q6. Tuning hybrid search

In [ ]:
k_values = [1, 50, 100, 200]

for k in k_values:
    result = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    print(f"k={k}: {result}")